In [1]:
# Step 1: Install required packages
!pip install -q transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# Step 3: Parse CoNLL format
def read_conll(file_path):
    sentences = []
    labels = []
    with open(file_path, encoding='utf-8') as f:
        tokens = []
        tags = []
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sentences.append(tokens)
                    labels.append(tags)
                    tokens, tags = [], []
            else:
                token, tag = line.split()
                tokens.append(token)
                tags.append(tag)
    return sentences, labels

# Replace this path with your uploaded file path
file_path = "/content/amharic_ner_subset.conll"
sentences, tags = read_conll(file_path)

In [3]:
# Step 4: Convert to Hugging Face Dataset format
from datasets import Dataset

dataset = Dataset.from_dict({
    "tokens": sentences,
    "ner_tags": tags
})
label_list = list(set(tag for seq in tags for tag in seq))
label_list.sort()
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}

In [8]:
# Step 5: Tokenize and align labels
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("xlm-roberta-base")

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True, padding=True, is_split_into_words=True)
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        label_ids = []
        prev_word_idx = None
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            else:
                label_str = label[word_idx]
                if word_idx != prev_word_idx:
                    label_ids.append(label2id[label_str])
                else:
                    if label_str.startswith("B-"):
                        label_str = label_str.replace("B-", "I-")
                    label_ids.append(label2id[label_str])
                prev_word_idx = word_idx
        labels.append(label_ids)
    tokenized["labels"] = labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)

Map:   0%|          | 0/39 [00:00<?, ? examples/s]

In [9]:
# Step 6: Fine-tune model
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./ner_model",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    save_steps=10_000,
    logging_dir="./logs",
    logging_steps=50,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer,
)

trainer.train()

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-9-1515488681.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss


TrainOutput(global_step=15, training_loss=0.7322900772094727, metrics={'train_runtime': 432.9785, 'train_samples_per_second': 0.27, 'train_steps_per_second': 0.035, 'total_flos': 15167125079964.0, 'train_loss': 0.7322900772094727, 'epoch': 3.0})

In [10]:
trainer.save_model("/content/amharic_ner_model")
tokenizer.save_pretrained("/content/amharic_ner_model")

('/content/amharic_ner_model/tokenizer_config.json',
 '/content/amharic_ner_model/special_tokens_map.json',
 '/content/amharic_ner_model/sentencepiece.bpe.model',
 '/content/amharic_ner_model/added_tokens.json',
 '/content/amharic_ner_model/tokenizer.json')